In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import mat73
from tqdm.auto import tqdm
from PIL import Image
import h5py
from scipy.stats import zscore, pearsonr

from sklearn.model_selection import train_test_split

In [ ]:
from brainscore_vision import load_dataset, load_benchmark
# from brainscore_vision.benchmark_helpers.neural_common import apply_keep_attrs

In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
NOISE_CEILING_THRESHOLD = 10

In [ ]:

def load_FZ(region='V1', avg_time_window=False, time_window=(50, 200)):
    assembly = load_dataset('FreemanZiemba2013.public')
    assembly = assembly.sel(region=region)
    assembly = assembly.stack(neuroid=['neuroid_id'])  # work around xarray multiindex issues
    assembly['region'] = 'neuroid', [region] * len(assembly['neuroid'])
    assembly.load()

    if avg_time_window:
        assembly = assembly.sel(time_bin=[(t, t + 1) for t in range(*time_window)])
        assembly = assembly.mean(dim='time_bin', keep_attrs=True)
        assembly = assembly.expand_dims('time_bin_start').expand_dims('time_bin_end')
        assembly['time_bin_start'], assembly['time_bin_end'] = [time_window[0]], [time_window[1]]
        assembly = assembly.stack(time_bin=['time_bin_start', 'time_bin_end'])
        assembly = assembly.squeeze('time_bin')
        assembly = assembly.transpose('presentation', 'neuroid')
        

    # assembly: DataArray with dims ('time_bin','presentation','neuroid')
    # flatten the MultiIndex -> turn its levels into coords
    da = assembly.reset_index('presentation')  # now coords like 'repetition','image_id', 'stimulus_id', ...

    # build (stimulus, repetition) index and unstack
    da_sr = (
        da
        .set_index(presentation=[ 'stimulus_id', 'repetition'])
        .unstack('presentation')   # -> dims: time_bin, neuroid, 'stimulus_id', 'repetition'
    )

    # reorder to (neurons, time_bins, stimuli, repetitions) and export
    if 'time_bin' in da_sr.dims:
        da_n_t_s_r = da_sr.transpose('neuroid', 'time_bin', 'stimulus_id', 'repetition')
    else:
        da_n_t_s_r = da_sr.transpose('neuroid', 'stimulus_id', 'repetition')
    assembly = assembly.sortby('stimulus_id')
    da_n_t_s_r = da_n_t_s_r.sortby('stimulus_id')
    arr = da_n_t_s_r.to_numpy()  # shape: (N_neurons, N_time_bins, N_stimuli, N_reps)

    return arr, assembly, da_n_t_s_r


In [ ]:
array_V1, assembly_V1, da_n_t_s_r_V1 = load_FZ('V1', avg_time_window=True)
array_V1 = array_V1.transpose(1, 0, 2)  # (stimuli, neurons, time_bins)

In [ ]:
array_V2, assembly_V2, da_n_t_s_r_V2 = load_FZ('V2', avg_time_window=True)
array_V2 = array_V2.transpose(1, 0, 2)  # (stimuli, neurons, time_bins)

In [ ]:
assembly_V1.attrs['stimulus_set']

In [ ]:
array_V1.shape, array_V2.shape

In [ ]:
SUBJECTS = ["monkeys"]
ROIS = ["V1", "V2"]

In [ ]:

stimulus_ids_V1 = assembly_V1.attrs['stimulus_set'].sort_values('stimulus_id').stimulus_id.values
stimulus_ids_V2 = assembly_V2.attrs['stimulus_set'].sort_values('stimulus_id').stimulus_id.values
texture_types = assembly_V1.attrs['stimulus_set'].sort_values('stimulus_id').texture_type.values

assert np.array_equal(stimulus_ids_V1, stimulus_ids_V2)
assert np.array_equal(da_n_t_s_r_V1.stimulus_id.values, stimulus_ids_V1)
assert np.array_equal(da_n_t_s_r_V2.stimulus_id.values, stimulus_ids_V2)


In [ ]:
indices = assembly_V1.attrs['stimulus_set'].index.values
indices_train, indices_test = train_test_split(indices, stratify=texture_types, test_size=0.1, random_state=42, shuffle=True)

stimulus_ids_train, stimulus_ids_test = stimulus_ids_V1[indices_train], stimulus_ids_V1[indices_test]


In [ ]:


noise_ceilings_variancebased = {
    "monkeys": {
        "V1": compute_ceiling_variancebased(array_V1.transpose(1, 0, 2)),  # (neurons, stimuli, repetitions)
        "V2": compute_ceiling_variancebased(array_V2.transpose(1, 0, 2)),  # (neurons, stimuli, repetitions)
    }
}

noise_ceilings_splithalf = {
    "monkeys": {
        "V1": compute_ceiling_splithalf(array_V1.transpose(1, 0, 2)).mean(axis=-1),
        "V2": compute_ceiling_splithalf(array_V2.transpose(1, 0, 2)).mean(axis=-1),
    }
}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i, roi in enumerate(ROIS):
    sns.histplot(noise_ceilings_variancebased['monkeys'][roi], bins=20, ax=axes[i], color='blue', label='Variance-based')
    sns.histplot(noise_ceilings_splithalf['monkeys'][roi], bins=20, ax=axes[i], color='orange', label='Split-half')
    axes[i].set_title(f'Noise Ceiling Distribution - {roi}')
    axes[i].set_xlabel('Noise Ceiling')
    axes[i].set_ylabel('Number of Neurons')
    axes[i].legend()
plt.tight_layout()

In [ ]:
valied_neuroids = {"monkeys": {
        "V1": np.where(noise_ceilings_variancebased['monkeys']['V1'] >= NOISE_CEILING_THRESHOLD)[0],
        "V2": np.where(noise_ceilings_variancebased['monkeys']['V2'] >= NOISE_CEILING_THRESHOLD)[0],
    }}



In [ ]:


noise_ceilings_variancebased_filtered = {
    "monkeys": {
        "V1": noise_ceilings_variancebased['monkeys']['V1'][valied_neuroids['monkeys']['V1']],
        "V2": noise_ceilings_variancebased['monkeys']['V2'][valied_neuroids['monkeys']['V2']],
    }
}

noise_ceilings_splithalf_filtered = {
    "monkeys": {
        "V1": noise_ceilings_splithalf['monkeys']['V1'][valied_neuroids['monkeys']['V1']],
        "V2": noise_ceilings_splithalf['monkeys']['V2'][valied_neuroids['monkeys']['V2']],
    }
}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
bins = np.linspace(0, 100, 21)
for i, roi in enumerate(ROIS):
    sns.histplot(noise_ceilings_variancebased['monkeys'][roi], bins=bins, ax=axes[i], color='blue', label='Variance-based')
    sns.histplot(noise_ceilings_splithalf['monkeys'][roi], bins=bins, ax=axes[i], color='orange', label='Split-half')
    axes[i].set_title(f'Noise Ceiling Distribution - {roi}')
    axes[i].set_xlabel('Noise Ceiling')
    axes[i].set_ylabel('Number of Neurons')
    axes[i].legend()
plt.tight_layout()

In [ ]:
for subject in SUBJECTS:
    for roi in ROIS:
        num_total = array_V1.shape[1] if roi == 'V1' else array_V2.shape[1]
        num_valid = len(valied_neuroids[subject][roi])
        print(f"{subject} - {roi}: {num_valid}/{num_total} neurons passed the noise ceiling threshold of {NOISE_CEILING_THRESHOLD}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
bins = np.linspace(0, 100, 21)
for i, roi in enumerate(ROIS):
    sns.histplot(noise_ceilings_variancebased_filtered['monkeys'][roi], bins=bins, ax=axes[i], color='blue', label='Variance-based')
    sns.histplot(noise_ceilings_splithalf_filtered['monkeys'][roi], bins=bins, ax=axes[i], color='orange', label='Split-half')
    axes[i].set_title(f'Noise Ceiling Distribution - {roi}')
    axes[i].set_xlabel('Noise Ceiling')
    axes[i].set_ylabel('Number of Neurons')
    axes[i].legend()
plt.tight_layout()

In [ ]:
subject_train_data = {
    'monkeys': {
        'V1': array_V1[indices_train][:, valied_neuroids['monkeys']['V1']].mean(axis=-1),  # average over repetitions
        'V2': array_V2[indices_train][:, valied_neuroids['monkeys']['V2']].mean(axis=-1),
    }
}

subject_test_data = {
    'monkeys': {
        'V1': array_V1[indices_test][:, valied_neuroids['monkeys']['V1']].mean(axis=-1),  # average over repetitions
        'V2': array_V2[indices_test][:, valied_neuroids['monkeys']['V2']].mean(axis=-1),
    }
}

In [ ]:
for sub in SUBJECTS:
    for roi in ROIS:
        print(f"{sub} {roi} train data shape: {subject_train_data[sub][roi].shape}")
        print(f"{sub} {roi} test data shape: {subject_test_data[sub][roi].shape}")

In [ ]:
for region in ['V1', 'V2']:
    print(noise_ceilings_variancebased['monkeys'][region].mean(), noise_ceilings_splithalf['monkeys'][region].mean())

In [ ]:
for subject in SUBJECTS:
    for roi in ROIS:
        assert noise_ceilings_variancebased_filtered[subject][roi].shape[0] == subject_train_data[subject][roi].shape[1]
        assert noise_ceilings_variancebased_filtered[subject][roi].shape[0] == subject_test_data[subject][roi].shape[1]

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": stimulus_ids_train,
            "neural_data": subject_train_data,
        },
    "test" :
        {
            "stimulus_ids": stimulus_ids_test,
            "neural_data": subject_test_data,
        },
    "noise_ceilings": noise_ceilings_variancebased_filtered,
}

### Metadata

In [ ]:
metadata = {
    "desc": f"""
    Freeman & Ziemba 2013 dataset processed.
    10% of the stimuli are held out as test set.
    The neural data is averaged over repetitions and over a time window of 50-200ms post stimulus onset.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = 'bs_fz.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi in ROIS:
                f.create_dataset(f"{split}/neural_data/{subj}/{roi}", data=processed_data[split]['neural_data'][subj][roi])
                
    for subj in SUBJECTS:
        for roi in ROIS:
            f.create_dataset(f"noise_ceilings/{subj}/{roi}", data=processed_data['noise_ceilings'][subj][roi])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['monkeys']['V1'].shape, loaded_data['noise_ceilings']['monkeys']['V1'].shape

In [ ]:

# Check the saved data
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    print(f.keys())
    for split in splits:
        print(f[split]['stimulus_ids'].shape)

        for subj in subjects:
            for region in rois:
                print(f[split]['neural_data'][subj][region].shape)
            
    print(json.loads(f.attrs['metadata']))